# 03. A2A (Agent2Agent) with LangSmith / Agent Server

> **주제**: 분산된 에이전트끼리 *서로 메시지를 주고받기* + 분산 트레이싱
>
> **원문**: https://docs.langchain.com/langsmith/server-a2a

---

## 이 노트북에서 배우는 것
1. A2A 프로토콜과 엔드포인트(`/a2a/{assistant_id}`), 지원 RPC 메서드
2. A2A 호환 에이전트의 **필수 조건**(state 에 `messages` 키)
3. `contextId` / `taskId` 로 대화 스레드 이어가기
4. 에이전트 ↔ 에이전트 대화 시뮬레이션
5. `contextId → thread_id` 매핑을 통한 LangSmith 분산 트레이싱

## 1. A2A 한눈에 보기

**A2A(Agent2Agent)** 는 *서로 다른 곳에 배포된 에이전트*들이 표준 메시징으로 협업하게 하는 프로토콜입니다. LangSmith 의 Agent Server 가 A2A 를 기본 지원하며, 대화 전반의 **분산 트레이싱**까지 유지합니다.

- **세 프로토콜 비교**: MCP(에이전트↔도구), ACP(에이전트↔에디터), **A2A(에이전트↔에이전트)**

**엔드포인트**: `POST /a2a/{assistant_id}` — JSON-RPC 2.0

**지원 RPC 메서드**
| 메서드 | 설명 |
|--------|------|
| `message/send` | 메시지 전송 후 완성된 응답 수신 |
| `message/stream` | SSE 로 실시간 스트리밍 |
| `tasks/get` | 작업 상태 확인 / 이전 결과 조회 |

**Agent Card 디스커버리**: `GET /.well-known/agent-card.json?assistant_id={assistant_id}` → 이름·설명·skills·I/O 모드·A2A 엔드포인트 반환

In [ ]:
# A2A 는 langgraph-api 0.4.21 이상이 필요 (uv)
!uv pip install -q "langgraph-api>=0.4.21"
# 클라이언트 예제용
!uv pip install -q aiohttp openai

## 2. A2A 호환 에이전트의 필수 조건

> **핵심**: 에이전트 state 에 **`messages` 키**가 있어야 A2A 의 text part 를 처리할 수 있습니다.

아래는 가장 단순한 state 정의입니다. `Context` 는 configurable 파라미터용, `State` 는 대화 상태용입니다.

In [ ]:
from typing import Any, Dict, List, TypedDict
from dataclasses import dataclass

class Context(TypedDict):
    my_configurable_param: str

@dataclass
class State:
    messages: List[Dict[str, Any]]   # <- A2A text part 처리를 위해 필수

## 3. 완전한 에이전트 구현 (graph.py)

LangGraph 그래프 하나를 정의합니다. `call_model` 노드가 마지막 사용자 메시지를 받아 OpenAI 로 응답을 만들고, 누적된 `messages` 에 덧붙입니다.

이 파일을 `langgraph.json` 에 등록하면 Agent Server 가 자동으로 `/a2a/{assistant_id}` 엔드포인트를 노출합니다.

In [ ]:
graph_py = '''
from __future__ import annotations
import os
from dataclasses import dataclass
from typing import Any, Dict, List, TypedDict

from langgraph.graph import StateGraph
from langgraph.runtime import Runtime
from openai import AsyncOpenAI

class Context(TypedDict):
    my_configurable_param: str

@dataclass
class State:
    messages: List[Dict[str, Any]]

async def call_model(state: State, runtime: Runtime[Context]) -> Dict[str, Any]:
    client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    latest_message = state.messages[-1] if state.messages else {}
    user_content = latest_message.get("content", "No message content")

    openai_messages = [
        {"role": "system",
         "content": "You are a helpful conversational agent. Keep responses brief and engaging."},
        {"role": "user", "content": user_content},
    ]

    try:
        response = await client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=openai_messages,
            max_tokens=100,
            temperature=0.7,
        )
        ai_response = response.choices[0].message.content
    except Exception as e:
        ai_response = f"I received your message but had trouble processing it. Error: {str(e)[:50]}..."

    response_message = {"role": "assistant", "content": ai_response}
    return {"messages": state.messages + [response_message]}

graph = (
    StateGraph(State, context_schema=Context)
    .add_node(call_model)
    .add_edge("__start__", "call_model")
    .compile()
)
'''

with open("graph.py", "w") as f:
    f.write(graph_py)
print("graph.py 생성 — langgraph.json 에 등록 후 `langgraph dev` 로 서버 실행")

## 4. 대화 이어가기: `contextId` 와 `taskId`

A2A 는 두 개의 식별자로 대화를 관리합니다.

- **`contextId`**: 메시지들을 하나의 **대화 스레드**로 묶음 (세션 ID 에 해당)
- **`taskId`**: 대화 안의 **개별 요청**을 식별

> **규칙**: 첫 메시지에는 둘 다 **생략** → 서버가 생성해서 돌려줌. 이후 메시지엔 둘 다 **포함**해서 스레드를 이어감.

## 5. 에이전트 ↔ 에이전트 대화 시뮬레이션

두 에이전트(A: 포트 2024, B: 포트 2025)를 번갈아 호출해 한 에이전트의 응답을 다른 에이전트의 입력으로 넘기는 핑퐁 대화입니다.

`extract_text()` 는 A2A 응답에서 텍스트를 꺼내는 헬퍼입니다 — 응답 텍스트는 `artifacts[].parts[]` 또는 `status.message.parts[]` 에 들어 있습니다.

In [ ]:
#!/usr/bin/env python3
import asyncio
import aiohttp
import os
import uuid

def extract_text(result: dict) -> str:
    for art in result.get("result", {}).get("artifacts", []) or []:
        for part in art.get("parts", []) or []:
            if part.get("kind") == "text" and part.get("text"):
                return part["text"]

    msg = (result.get("result", {}).get("status", {}) or {}).get("message", {}) or {}
    for part in msg.get("parts", []) or []:
        if part.get("kind") == "text" and part.get("text"):
            return part["text"]

    return "(no text found)"

In [ ]:
async def send_message(session, port, assistant_id, text, context_id=None, task_id=None):
    url = f"http://127.0.0.1:{port}/a2a/{assistant_id}"

    message = {
        "role": "user",
        "parts": [{"kind": "text", "text": text}],
        "messageId": str(uuid.uuid4()),
    }
    if context_id:
        message["contextId"] = context_id
    if task_id:
        message["taskId"] = task_id

    payload = {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4()),
        "method": "message/send",
        "params": {"message": message},
    }

    headers = {"Accept": "application/json"}
    async with session.post(url, json=payload, headers=headers) as response:
        result = await response.json()

    returned_context_id = result.get("result", {}).get("contextId") or context_id
    returned_task_id = result.get("result", {}).get("id")
    return extract_text(result), returned_context_id, returned_task_id

In [ ]:
async def simulate_conversation():
    agent_a_id = os.getenv("AGENT_A_ID")
    agent_b_id = os.getenv("AGENT_B_ID")

    if not agent_a_id or not agent_b_id:
        print("Set AGENT_A_ID and AGENT_B_ID environment variables")
        return

    message = "Hello! Let's have a conversation."
    context_id = None
    task_id = None

    async with aiohttp.ClientSession() as session:
        for i in range(3):
            print(f"--- Round {i + 1} ---")

            message, context_id, task_id = await send_message(
                session, 2024, agent_a_id, message,
                context_id=context_id, task_id=task_id,
            )
            print(f"\U0001F535 Agent A: {message}")

            message, context_id, task_id = await send_message(
                session, 2025, agent_b_id, message,
                context_id=context_id, task_id=task_id,
            )
            print(f"\U0001F534 Agent B: {message}\n")

# 주피터에서는 await 로 실행 (스크립트면 asyncio.run(simulate_conversation()))
# await simulate_conversation()

## 6. LangSmith 분산 트레이싱

Agent Server 는 A2A 의 `contextId` 를 LangSmith 의 `thread_id` 로 **자동 매핑**합니다. 덕분에 여러 에이전트를 오가는 대화가 추가 설정 없이 하나의 thread 로 묶입니다.

여러 에이전트의 트레이스를 **하나로 통합**하려면:
1. 후속 턴 메시지에 `contextId` / `taskId` 포함
2. JSON-RPC 최상위 `metadata` 에 `thread_id` 전달
3. **모든 에이전트에서 같은 `thread_id` 재사용**

In [ ]:
import asyncio
import aiohttp
import uuid

async def send_message(session, url, text, context_id=None, task_id=None, thread_id=None):
    message = {
        "role": "user",
        "parts": [{"kind": "text", "text": text}],
        "messageId": str(uuid.uuid4()),
    }
    if context_id:
        message["contextId"] = context_id
    if task_id:
        message["taskId"] = task_id

    payload = {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4()),
        "method": "message/send",
        "params": {"message": message},
        "metadata": {"thread_id": thread_id},   # <- 트레이스 통합 키
    }

    async with session.post(url, json=payload, headers={"Accept": "application/json"}) as response:
        if response.status != 200:
            raise RuntimeError(f"HTTP {response.status}: {await response.text()}")
        result = await response.json()

    if "error" in result:
        raise RuntimeError(result["error"].get("message", "Unknown error"))

    result_obj = result.get("result", {})
    returned_context_id = result_obj.get("contextId") or context_id
    returned_task_id = result_obj.get("id")
    text_out = next(
        (
            part.get("text", "")
            for art in result_obj.get("artifacts", []) or []
            for part in art.get("parts", []) or []
            if part.get("kind") == "text"
        ),
        "(no text)",
    )
    return text_out, returned_context_id, returned_task_id

In [ ]:
async def run_conversation(agent_a_url, agent_b_url):
    thread_id = str(uuid.uuid4())
    context_id = None
    task_id = None
    message = "Hello! Let's collaborate."

    async with aiohttp.ClientSession() as session:
        for _ in range(3):
            message, context_id, task_id = await send_message(
                session, agent_a_url, message,
                context_id=context_id, task_id=task_id,
                thread_id=context_id or thread_id,
            )
            message, context_id, task_id = await send_message(
                session, agent_b_url, message,
                context_id=context_id, task_id=task_id,
                thread_id=context_id or thread_id,
            )

# asyncio.run(run_conversation(
#     "http://localhost:2024/a2a/<agent_a_assistant_id>",
#     "http://localhost:2025/a2a/<agent_b_assistant_id>",
# ))

## 7. 비-LangGraph 에이전트 트레이싱 & A2A 끄기

LangGraph 가 아닌 에이전트는 들어오는 A2A `metadata` 에서 `thread_id` 를 직접 꺼내 OTel span 에 붙입니다. 또 필요하면 `langgraph.json` 에서 A2A 를 끌 수 있습니다.

In [ ]:
# 비-LangGraph(FastAPI) 에이전트에서 thread_id 추출 → LangSmith 트레이싱
from fastapi import FastAPI, Request
from langsmith.integrations.otel import configure as configure_otel
from opentelemetry import trace
import json

configure_otel(project_name="my-a2a-project")
tracer = trace.get_tracer(__name__)

app = FastAPI()

@app.middleware("http")
async def set_thread_id_middleware(request: Request, call_next):
    thread_id = None
    if request.method == "POST":
        body_bytes = await request.body()
        if body_bytes:
            try:
                body = json.loads(body_bytes)
                thread_id = body.get("metadata", {}).get("thread_id")
            except Exception:
                pass
            async def receive():
                return {"type": "http.request", "body": body_bytes}
            request._receive = receive

    with tracer.start_as_current_span("agent") as span:
        if thread_id:
            span.set_attribute("langsmith.metadata.thread_id", thread_id)
        return await call_next(request)

# 환경변수 LANGSMITH_API_KEY, LANGSMITH_PROJECT 설정 필요

In [ ]:
# langgraph.json 에서 A2A 비활성화
disable_a2a = '''
{
  "$schema": "https://langgra.ph/schema.json",
  "http": {
    "disable_a2a": true
  }
}
'''
print(disable_a2a)

## 정리 & 연습 문제

**핵심 요약**
- A2A = **에이전트 ↔ 에이전트** 표준 메시징. 엔드포인트 `POST /a2a/{assistant_id}` (JSON-RPC 2.0)
- 메서드: `message/send`, `message/stream`, `tasks/get` / 디스커버리: `agent-card.json`
- 에이전트 state 에 **`messages` 키 필수**
- `contextId`(스레드) + `taskId`(요청)로 대화 이어감 — 첫 턴엔 생략, 이후 포함
- `contextId → thread_id` 자동 매핑 + `metadata.thread_id` 로 멀티 에이전트 트레이스 통합

**연습**
1. `graph.py` 의 system prompt 를 서로 다른 페르소나(낙관론자 vs 비관론자)로 둔 두 에이전트를 띄우고 핑퐁 대화를 돌려 보세요.
2. 같은 `thread_id` 로 호출한 뒤 LangSmith 에서 두 에이전트 트레이스가 한 thread 로 묶이는지 확인해 보세요.
3. `message/stream` 으로 바꿔 SSE 스트리밍 응답을 받아 보세요.

**참고 레포**
- 두 LangGraph 에이전트: https://github.com/langchain-samples/A2A-langgraph
- Google ADK + LangChain: https://github.com/langchain-samples/A2A-google-adk
- A2A 공식 명세: https://a2a-protocol.org/latest/